# Day 2, Session 2 -- Demos: LangChain Development & Tool Integration

This notebook contains all five instructor-led demos from Session 2. Run each cell, observe the output, and make sure you understand the pattern before moving on.

**Engineering context:** You are the engineer building LLM-powered applications with LangChain. Consultants are your users -- they need reusable analysis templates, composable pipelines, custom analytical tools, and knowledge retrieval systems they can query in natural language.

## Session Goal

Students will learn LangChain's core abstractions -- **ChatModels**, **PromptTemplates**, **LCEL chains**, **custom tools**, **document splitting**, and a **simple RAG pipeline**. These are the building blocks for the agent orchestration you will implement in Sessions 3-4.

By the end of this session you will be able to:
1. Initialize and invoke ChatModels with different message types
2. Build multi-step LCEL chains with different output parsers
3. Define custom tools and bind them to an LLM for tool selection
4. Split documents into overlapping chunks for retrieval
5. Build a complete (simple) RAG pipeline from scratch

In [ ]:
!pip install -q langchain langchain-openai langchain-core langchain-text-splitters python-dotenv

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import json
import os

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All imports successful!")

---
## Demo 1: LangChain Basics -- ChatModels and PromptTemplates

ChatModels wrap LLM APIs into a unified interface. PromptTemplates let you parameterize prompts with variables so they become reusable across different inputs.

**Scenario:** An engagement team needs to quickly generate structured analyses across different industries and strategy frameworks. PromptTemplates let us build reusable "analysis templates" that any consultant can invoke with different client parameters.

### Demo 1a: Initialize a ChatModel and Simple Invoke

The `ChatOpenAI` class wraps OpenAI's API. The key parameters are `model` (which model to use) and `temperature` (creativity vs determinism).

In [ ]:
# Demo 1a - Initialize ChatModel + simple invoke

# Initialize with environment variables for easy configuration
llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7"))
)

# Simple invocation with a plain string
response = llm.invoke("What is the MECE framework in one sentence?")
print("Simple invocation:")
print(f"  Type: {type(response)}")
print(f"  Content: {response.content}")
print(f"  Model: {response.response_metadata.get('model_name', 'N/A')}")

**Observe:** `llm.invoke()` returns an `AIMessage` object, not a plain string. Use `.content` to get the text. This object also carries metadata (model, usage stats). This is important because downstream components in LCEL chains expect `AIMessage` objects, not raw strings.

### Demo 1b: Invoke with Message Objects

Instead of a plain string, you can pass a list of typed message objects. This gives you explicit control over the system prompt and conversation history.

In [ ]:
# Demo 1b - Invoke with message objects

messages = [
    SystemMessage(content="You are a senior partner at McKinsey. Be concise and strategic."),
    HumanMessage(content="What is a value chain analysis?")
]

response = llm.invoke(messages)
print(f"With messages:\n  {response.content}")

### Demo 1c: PromptTemplates -- Reusable Parameterized Prompts

`ChatPromptTemplate` lets you define templates with `{variables}` that get filled at runtime. This is how you build reusable consulting analysis templates.

In [ ]:
# Demo 1c - PromptTemplates for reusable consulting prompts

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a consultant specializing in {domain}. Answer in {style} style."),
    ("human", "{question}")
])

# Fill the template with specific values
domain = "growth strategy and market entry"
style = "executive briefing"
question = "What are the key drivers of growth in the EV battery market?"

formatted = prompt.format_messages(
    domain=domain,
    style=style,
    question=question
)

print(f"Formatted prompt messages: {len(formatted)}")
for msg in formatted:
    print(f"  [{msg.type}]: {msg.content[:100]}...")

response = llm.invoke(formatted)
print(f"\nResponse: {response.content[:300]}...")

### Try This

Change the `domain` variable above to a different consulting specialty (e.g., `"digital transformation"`, `"supply chain optimization"`, `"M&A due diligence"`) and re-run the cell. Notice how the same template produces contextually different responses -- this is the power of parameterized prompts.

---
## Demo 2: Building Chains with LCEL (Pipe Operator)

LCEL (LangChain Expression Language) uses the `|` operator to compose components into chains. The pattern is: `prompt | model | parser`. Each component implements the Runnable interface, so they can be freely combined.

**Scenario:** Engagement teams need multi-step analysis pipelines: research a market, analyze competitive dynamics, then produce an executive summary. LCEL lets us chain these steps cleanly -- mirroring how a real consulting workflow flows from data gathering through synthesis to recommendation.

### Demo 2a: Simple Chain with StrOutputParser

The simplest chain: prompt template -> LLM -> string parser. `StrOutputParser()` extracts the `.content` from the `AIMessage` so you get a clean string.

In [ ]:
# Demo 2a - Simple chain with StrOutputParser

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5"))
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strategy consultant. Provide clear, structured insights."),
    ("human", "Explain {concept} in exactly {num_sentences} sentences, using a business consulting context.")
])

# The pipe operator composes: prompt -> llm -> parser
chain = prompt | llm | StrOutputParser()

# invoke() accepts a dict of template variables
result = chain.invoke({"concept": "Porter's Five Forces", "num_sentences": "3"})
print("Simple chain result:")
print(result)

### Demo 2b: Chain with JSON Output

`JsonOutputParser()` parses the LLM's string response into a Python dict. You instruct the LLM to output JSON in the system prompt.

In [ ]:
# Demo 2b - Chain with JSON output

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output a JSON object with keys: framework_name, definition, consulting_use_case, complexity (1-10). Output ONLY valid JSON, no markdown fences."),
    ("human", "Describe the consulting framework: {concept}")
])

json_chain = json_prompt | llm | JsonOutputParser()

result = json_chain.invoke({"concept": "MECE principle"})
print(f"JSON chain result (type={type(result).__name__}):")
print(json.dumps(result, indent=2))

**Key Insight:** `JsonOutputParser()` automatically parses the LLM's string output into a Python dict. If the LLM returns malformed JSON, this will raise an error -- catch it in production! You can also use Pydantic models with `JsonOutputParser(pydantic_object=MyModel)` for schema validation.

### Demo 2c: Multi-Step Chain (Summarize + Translate)

Sometimes you need to run multiple chains in sequence, passing the output of one as the input to the next.

In [ ]:
# Demo 2c - Multi-step chains: research -> executive summary -> translation

summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize this market research finding in one executive-ready sentence: {text}"
)
translate_prompt = ChatPromptTemplate.from_template(
    "Translate this executive summary to French for our Paris office: {text}"
)

summary_chain = summarize_prompt | llm | StrOutputParser()
translate_chain = translate_prompt | llm | StrOutputParser()

# Input: raw market research text
text = """The global electric vehicle battery market is projected to grow at a CAGR of 19.2% 
from 2024 to 2030. Key growth drivers include government incentives, declining lithium-ion 
costs, and rising consumer demand for sustainable transportation. Asia-Pacific dominates 
with 68% market share, led by CATL and BYD."""

# Step 1: Summarize
summary = summary_chain.invoke({"text": text})
print(f"Executive Summary: {summary}")

# Step 2: Translate (manually passing output of step 1)
translation = translate_chain.invoke({"text": summary})
print(f"\nFrench (for Paris office): {translation}")

**Gotcha:** Each chain is independent -- `translate_chain` doesn't know about `summary_chain`. You must manually pass the output of one as input to the next. LangGraph (Session 3) automates this with state, allowing you to define a graph where outputs flow automatically between nodes.

### QnA Recap -- Demo 2

Pause here for questions. Key points to revisit:

1. **Q: What does the `|` operator actually do?**  
   A: It creates a `RunnableSequence` -- each component's output becomes the next component's input. It is syntactic sugar for `.pipe()`.

2. **Q: When should I use `JsonOutputParser` vs `StrOutputParser`?**  
   A: Use `JsonOutputParser` when you need structured data (for downstream code logic). Use `StrOutputParser` when the output is meant for humans to read.

3. **Q: Can I chain more than 3 components?**  
   A: Yes! You can chain as many as needed: `prompt | llm | parser | next_prompt | llm | parser`. Each component just needs compatible input/output types.

---
## Demo 3: Creating and Using Custom Tools

Tools let LLMs interact with the real world. In LangChain, use the `@tool` decorator to turn any Python function into a tool that an LLM can discover and invoke. The docstring becomes the tool's description.

**Scenario:** Consultants rely on specialized analytical tools daily -- market sizing calculators, benchmark databases, financial ratio analyzers. Here we build custom tools that mirror these consulting utilities and bind them to an LLM so it can decide which tool to use based on a client question.

### Demo 3a: Define 3 Custom Tools

Each tool is a Python function decorated with `@tool`. The function signature defines the parameters, and the docstring is what the LLM reads to understand when to use the tool.

In [ ]:
# Demo 3a - Define custom consulting tools

@tool
def market_sizing_tool(population: float, adoption_rate: float, avg_revenue_per_user: float) -> dict:
    """Estimate total addressable market (TAM) given population, adoption rate (0-1), and average revenue per user."""
    tam = population * adoption_rate * avg_revenue_per_user
    return {
        "total_addressable_market": tam,
        "population": population,
        "adoption_rate_pct": f"{adoption_rate * 100:.1f}%",
        "arpu": avg_revenue_per_user,
        "formatted_tam": f"${tam:,.0f}"
    }

@tool
def benchmark_lookup_tool(metric: str) -> str:
    """Look up industry benchmark values for common consulting metrics like margins, growth rates, and R&D spend."""
    benchmarks = {
        "saas_gross_margin": "70-85% gross margin is typical for SaaS companies",
        "retail_operating_margin": "3-5% operating margin is typical for retail",
        "pharma_r_and_d_spend": "15-25% of revenue is typical pharma R&D spend",
        "tech_revenue_growth": "20-40% YoY revenue growth for high-growth tech",
        "healthcare_ebitda_margin": "15-25% EBITDA margin for healthcare services",
    }
    return benchmarks.get(metric.lower(), f"No benchmark found for '{metric}'. Available: {list(benchmarks.keys())}")

@tool
def financial_ratio_tool(revenue: float, costs: float, assets: float) -> dict:
    """Calculate key financial ratios (gross margin, asset turnover, profit) used in consulting engagements."""
    gross_margin = (revenue - costs) / revenue if revenue else 0
    asset_turnover = revenue / assets if assets else 0
    return {
        "gross_margin": f"{gross_margin:.1%}",
        "asset_turnover": f"{asset_turnover:.2f}x",
        "profit": f"${revenue - costs:,.0f}"
    }

print("3 tools defined: market_sizing_tool, benchmark_lookup_tool, financial_ratio_tool")

### Demo 3b: Inspect Tool Metadata and Invoke Directly

Before binding tools to an LLM, you can inspect their metadata and call them directly to verify they work.

In [ ]:
# Demo 3b - Inspect tool metadata + invoke directly

# Inspect what the LLM will "see" about this tool
print("Tool: market_sizing_tool")
print(f"  Name: {market_sizing_tool.name}")
print(f"  Description: {market_sizing_tool.description}")
print(f"  Args schema: {market_sizing_tool.args}")

print("\n" + "="*60)
print("Direct invocations (no LLM involved):")
print("="*60)

# Invoke tools directly -- useful for testing
result1 = market_sizing_tool.invoke({
    "population": 330_000_000,
    "adoption_rate": 0.08,
    "avg_revenue_per_user": 45000
})
print(f"\nmarket_sizing_tool (US EV market):")
print(f"  {result1}")

result2 = benchmark_lookup_tool.invoke("saas_gross_margin")
print(f"\nbenchmark_lookup_tool:")
print(f"  {result2}")

result3 = financial_ratio_tool.invoke({
    "revenue": 50_000_000,
    "costs": 32_000_000,
    "assets": 80_000_000
})
print(f"\nfinancial_ratio_tool:")
print(f"  {result3}")

### Demo 3c: Bind Tools to an LLM

When you bind tools to an LLM, the model can decide WHICH tool to call (if any) based on the user's question. The LLM does NOT execute the tool -- it only returns the tool name and arguments.

In [ ]:
# Demo 3c - Bind tools to an LLM

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
)

# bind_tools() attaches tool schemas to the LLM
llm_with_tools = llm.bind_tools([market_sizing_tool, benchmark_lookup_tool, financial_ratio_tool])

# Ask a question that should trigger a tool call
response = llm_with_tools.invoke("What is the typical gross margin for a SaaS company?")
print("LLM response with tools bound:")
print(f"  Content: '{response.content}'")
print(f"  Tool calls: {response.tool_calls}")

# Ask another question
response2 = llm_with_tools.invoke("Calculate market size for 5 million users at 10% adoption and $200 ARPU")
print(f"\nSecond query tool calls: {response2.tool_calls}")

**Observe:** When the LLM decides to use a tool, `response.content` is empty and `response.tool_calls` contains the tool name and arguments. Your code is responsible for executing the tool -- the LLM just decides WHICH tool to call. This is the foundation of agent architectures: the LLM is the "brain" that routes to tools, and your code is the "hands" that execute them.

### Try This

Add a 4th tool -- for example, a `growth_rate_tool(current_value, previous_value)` that calculates year-over-year growth rate. Define it with `@tool`, add it to the `bind_tools()` list, and ask the LLM a question that should trigger it.

### QnA Recap -- Demo 3

Pause here for questions. Key points to revisit:

1. **Q: Does the LLM actually run the tool?**  
   A: No! The LLM only decides which tool to call and what arguments to pass. Your code must execute the tool and (optionally) pass the result back to the LLM.

2. **Q: What if the LLM picks the wrong tool?**  
   A: Better docstrings help. You can also add few-shot examples or use a system prompt that describes when each tool is appropriate.

3. **Q: Can the LLM call multiple tools at once?**  
   A: Yes! Some models support parallel tool calling. Check `response.tool_calls` -- it can contain multiple entries.

---
## Demo 4: Document Loading and Text Splitting

RAG starts with loading documents and splitting them into manageable chunks. The `RecursiveCharacterTextSplitter` splits at natural boundaries (paragraphs, sentences) and maintains overlap between chunks so context is not lost at boundaries.

**Scenario:** Consulting firms maintain vast knowledge bases -- industry reports, client case studies, best practice documents. Before we can retrieve relevant insights, we need to split these documents into chunks that fit within LLM context windows while preserving analytical coherence.

### Demo 4a: Create Sample Documents

In LangChain, a `Document` object holds `page_content` (the text) and `metadata` (source, chapter, date, etc.).

In [ ]:
# Demo 4a - Create sample consulting documents

documents = [
    Document(
        page_content="""McKinsey Global Institute research shows that generative AI could add $2.6 to $4.4 trillion
annually to the global economy across 63 use cases. The banking, high-tech, and life sciences
sectors stand to benefit most, with potential productivity gains of 3-5% of total industry revenue.
Organizations that move quickly on adoption will capture disproportionate value.""",
        metadata={"source": "mgi_genai_report.pdf", "chapter": "Executive Summary", "year": 2024}
    ),
    Document(
        page_content="""Post-merger integration (PMI) remains the most critical phase of any M&A transaction.
McKinsey research indicates that 70% of mergers fail to achieve expected synergies, primarily due to
cultural misalignment and slow integration execution. Best-practice PMI follows a 100-day plan
covering Day 1 readiness, synergy capture, operating model design, and cultural integration.""",
        metadata={"source": "mckinsey_ma_playbook.pdf", "chapter": "Post-Merger Integration", "year": 2023}
    )
]

print(f"Loaded {len(documents)} consulting documents")
for doc in documents:
    print(f"  Source: {doc.metadata['source']} | Chapter: {doc.metadata['chapter']} | Length: {len(doc.page_content)} chars")

### Demo 4b: Split Documents and Show Chunks + Overlap

The `RecursiveCharacterTextSplitter` tries to split at natural boundaries in this priority order: `\n\n` (paragraphs) > `\n` (lines) > `. ` (sentences) > ` ` (words) > `""` (characters).

In [ ]:
# Demo 4b - Split documents into chunks and show overlap

splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "200")),
    chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")),
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(chunks)} chunks:")
print(f"  chunk_size=200, chunk_overlap=30")

for i, chunk in enumerate(chunks):
    print(f"\n  Chunk {i+1} ({len(chunk.page_content)} chars) [from {chunk.metadata['source']}]:")
    print(f"    '{chunk.page_content[:100]}...'")

# Show how overlap works between consecutive chunks from the same document
print("\n" + "="*60)
print("OVERLAP DEMONSTRATION:")
print("="*60)
print(f"  End of chunk 1  : ...'{chunks[0].page_content[-50:]}'")
if len(chunks) > 1:
    print(f"  Start of chunk 2: '{chunks[1].page_content[:50]}...'")
    
    # Find the overlapping text
    end_text = chunks[0].page_content[-30:]
    start_text = chunks[1].page_content[:50]
    print(f"\n  Look for the overlap: text that appears in BOTH chunks.")

**Key Insight:** Chunk overlap prevents information loss at boundaries. If a key fact spans two chunks, the overlap ensures at least one chunk contains the complete fact. The trade-off: more overlap = more redundancy = more tokens to process, but fewer lost facts.

---
## Demo 5: Building a Simple RAG Pipeline

This demo puts it all together: a knowledge base of text chunks, a simple retrieval function, and an LCEL chain that generates answers grounded in retrieved context.

**Scenario:** Imagine a knowledge management system where consultants can query the firm's repository of industry insights, frameworks, and engagement best practices. The RAG pipeline retrieves the most relevant knowledge snippets and generates a grounded, evidence-based answer -- just like a seasoned partner drawing on decades of firm knowledge.

### Demo 5a: Knowledge Base and Retrieval Function

We create a simple keyword-based retriever. In Day 3, you will replace this with embedding-based semantic retrieval.

In [ ]:
# Demo 5a - Knowledge base + retrieval function

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
)

# Consulting knowledge base (in production, these would be vector-embedded chunks)
knowledge_base = [
    "The MECE framework (Mutually Exclusive, Collectively Exhaustive) is a foundational structuring principle. It ensures problem decomposition has no overlaps and no gaps, enabling rigorous hypothesis-driven analysis.",
    "The Three Horizons of Growth framework helps companies allocate resources: Horizon 1 (core business), Horizon 2 (emerging opportunities), Horizon 3 (transformational initiatives).",
    "Value chain analysis, developed by Michael Porter, maps the primary and support activities that create value for customers. Consultants use it to identify cost reduction and differentiation opportunities.",
    "The 7S Framework examines seven interdependent elements: Strategy, Structure, Systems, Shared Values, Style, Staff, and Skills. It is used for organizational effectiveness assessments.",
    "Digital transformation engagements typically follow a 'rewire' approach: reimagine the business model, rebuild the technology foundation, and rewire the organization for speed and agility.",
    "Post-merger integration (PMI) success depends on capturing synergies within the first 100 days. The PMI approach covers Day 1 readiness, clean rooms, synergy tracking, and cultural integration.",
]

# Simple keyword-based retrieval (counts word overlap)
def simple_retrieve(query, k=2):
    """Retrieve top-k chunks by keyword overlap with the query."""
    scored = []
    query_words = set(query.lower().split())
    for chunk in knowledge_base:
        chunk_words = set(chunk.lower().split())
        overlap = len(query_words & chunk_words)
        scored.append((overlap, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [chunk for _, chunk in scored[:k]]

# Test retrieval
test_query = "What is the MECE framework?"
retrieved = simple_retrieve(test_query)
print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved)} chunks:")
for i, chunk in enumerate(retrieved):
    print(f"  {i+1}. {chunk[:80]}...")

### Demo 5b: RAG Chain -- Retrieval + Generation

Now we connect retrieval to generation: retrieve relevant chunks, inject them as context, and ask the LLM to answer based ONLY on that context.

In [ ]:
# Demo 5b - RAG chain + run queries

# RAG prompt: instruct the LLM to answer ONLY from context
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a knowledge management assistant for a consulting firm. 
Answer based ONLY on the provided context from the knowledge base. 
If the answer is not found in the context, say 'This topic is not covered in the current knowledge base.'

Context:
{context}"""),
    ("human", "{question}")
])

rag_chain = rag_prompt | llm | StrOutputParser()

# Run consulting queries
questions = [
    "What is the MECE framework?",
    "How does post-merger integration work?",
    "What is the capital asset pricing model?"
]

for question in questions:
    # Step 1: Retrieve relevant context
    retrieved = simple_retrieve(question)
    context = "\n".join(f"- {chunk}" for chunk in retrieved)
    
    # Step 2: Generate answer grounded in context
    answer = rag_chain.invoke({"context": context, "question": question})
    
    print(f"Q: {question}")
    print(f"A: {answer}")
    print()

**Gotcha:** Our keyword retrieval is naive -- it misses semantic matches. The query 'capital asset pricing model' correctly returns no results because it is not in our knowledge base. But even if it WERE, keyword matching would struggle with synonyms and paraphrases. In Day 3, you will use embeddings for semantic retrieval that understands meaning, not just word overlap.

### Try This

Add a new document to the `knowledge_base` list -- for example, a paragraph about the BCG Growth-Share Matrix or Blue Ocean Strategy. Then re-run the retrieval and ask a question about your new topic. Does it get retrieved correctly?

### QnA Recap -- Demo 5

Pause here for questions. Key points to revisit:

1. **Q: Why does the RAG prompt say 'answer ONLY from context'?**  
   A: This prevents hallucination. Without this instruction, the LLM would draw on its training data, which may be outdated or inaccurate for your specific domain.

2. **Q: What happens if retrieval returns irrelevant chunks?**  
   A: The LLM may generate an incorrect answer based on irrelevant context. Retrieval quality is the #1 factor in RAG performance -- garbage in, garbage out.

3. **Q: How do we improve beyond keyword matching?**  
   A: Use embedding-based retrieval (Day 3). You convert text to vectors and find semantically similar chunks, even if they use different words.

---
## Summary

In this demo notebook you saw five core LangChain building blocks:

1. **ChatModels & PromptTemplates** -- Unified LLM interface and reusable, parameterized prompt templates.
2. **LCEL Chains** -- The pipe operator (`|`) composes prompt, model, and parser into clean pipelines.
3. **Custom Tools** -- The `@tool` decorator turns Python functions into tools LLMs can discover and invoke.
4. **Document Loading & Splitting** -- Preparing knowledge base documents for retrieval by splitting into overlapping chunks.
5. **RAG Pipelines** -- Combining retrieval with generation to ground answers in external knowledge.

**Next:** Open `D2S2_exercises.ipynb` to practice building these components yourself -- from custom tools to full RAG pipelines.